<a href="https://colab.research.google.com/github/nategrod07/nategrod07/blob/main/I310D_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import seaborn
import matplotlib
import sklearn

# Download latest version
path = kagglehub.dataset_download("psparks/instacart-market-basket-analysis")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'instacart-market-basket-analysis' dataset.
Path to dataset files: /kaggle/input/instacart-market-basket-analysis


In [ ]:
#PHASE 1 : DATA CLEANING
dataframes = {}
CSV_FILES = [
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv",
]

#iterate through files

for filename in CSV_FILES:
    key = filename.replace(".csv", "")
    dataframes[key] = pd.read_csv(f"/content/{filename}")
    print(f"Loaded {filename}")
#

for name, df in dataframes.items():
    out_path = f"/content/{name}_clean.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved {name}_clean.csv")

Loaded orders.csv
Loaded order_products__prior.csv
Loaded order_products__train.csv
Loaded products.csv
Loaded aisles.csv
Loaded departments.csv
Saved orders_clean.csv
Saved order_products__prior_clean.csv
Saved order_products__train_clean.csv
Saved products_clean.csv
Saved aisles_clean.csv
Saved departments_clean.csv


In [ ]:
#PHASE 2 : DATA MERGING

orders = pd.read_csv("/content/orders_clean.csv")
order_products_prior = pd.read_csv("/content/order_products__prior_clean.csv")
products = pd.read_csv("/content/products_clean.csv")

print(f"  Loaded orders:                {orders.shape[0]:,} rows x {orders.shape[1]} columns")
print(f"  Loaded order_products__prior: {order_products_prior.shape[0]:,} rows x {order_products_prior.shape[1]} columns")
print(f"  Loaded products:              {products.shape[0]:,} rows x {products.shape[1]} columns")

#merging order_product and prior is necessary for the logistical regression model


order_product_df = order_products_prior.merge(
    orders[["order_id", "user_id", "order_number", "days_since_prior_order"]],
    on="order_id",
    how="left"
)

print(f"  Rows after merge: {order_product_df.shape[0]:,}")
print(f"  Columns: {list(order_product_df.columns)}")
print(f"\n  Sample rows:")
print(order_product_df.head(5).to_string())

#Descriptive statistics


print(f"\n  Unique users:    {order_product_df['user_id'].nunique():,}")
print(f"  Unique orders:   {order_product_df['order_id'].nunique():,}")
print(f"  Unique products: {order_product_df['product_id'].nunique():,}")

basket_size = order_product_df.groupby("order_id")["product_id"].count()
print(f"\n  Avg basket size: {basket_size.mean():.2f} items per order")
print(f"  Min basket size: {basket_size.min()}")
print(f"  Max basket size: {basket_size.max()}")

print(f"\n  days_since_prior_order stats:")
print(order_product_df["days_since_prior_order"].describe().to_string())

reorder_rate = order_product_df["reordered"].mean() * 100
print(f"\n  Overall reorder rate: {reorder_rate:.2f}%")

#Save — order_product_df for churn labeling and features and keep products
#seperate to be joined later on for product name reporting

order_product_df.to_csv("/content/order_product_df.csv", index=False)
print("  Saved order_product_df.csv  — used in Phases 4, 5, and 6")
print("  products_clean.csv          — will be joined in Phase 5 only for product names")


  Loaded orders:                3,421,083 rows x 7 columns
  Loaded order_products__prior: 32,434,489 rows x 4 columns
  Loaded products:              49,688 rows x 4 columns
  Rows after merge: 32,434,489
  Columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'order_number', 'days_since_prior_order']

  Sample rows:
   order_id  product_id  add_to_cart_order  reordered  user_id  order_number  days_since_prior_order
0         2       33120                  1          1   202279             3                     8.0
1         2       28985                  2          1   202279             3                     8.0
2         2        9327                  3          0   202279             3                     8.0
3         2       45918                  4          1   202279             3                     8.0
4         2       30035                  5          0   202279             3                     8.0

  Unique users:    206,209
  Unique orders:   

In [ ]:
#PHASE 3 : CHURN DEFINITION

In [ ]:
#PHASE 4 : PRODUCT LEVEL CHURN ANALYSIS